In [ ]:
!pip install --user torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
pip install numba

In [ ]:
from numba import jit, cuda

In [ ]:
pip install langchain

In [ ]:
pip install accelerate

In [ ]:
pip install bitsandbytes

In [ ]:
from langchain.llms import HuggingFacePipeline
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, AutoModelForSeq2SeqLM

In [ ]:
from langchain import PromptTemplate, HuggingFaceHub, LLMChain

template = """You are tasked with generating a brief elaboration of one sentence from a given context. The context will be in the form of a paragraph consisting of multiple sentences. 
You're required to generate an appropriate new sentence that adds to the reader's understanding of the context in a clear and coherent way. 
Ensure the new sentence is concise and directly relevant to the information presented. Here is the context: {sentence}

Answer: """

prompt = PromptTemplate(template=template, input_variables=["sentence"])

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

In [ ]:
print(torch.cuda.is_available())

In [ ]:
import pandas as pd

df = pd.read_csv('elaboration_data.csv',header=0)



In [ ]:
df.info()

In [ ]:
print(df.select_idx)

In [ ]:
context_df = df.loc[df['elaboration_type'] == 'context-only'].context
context_df = context_df.replace('\n', ' ', regex=True)
print(context_df.loc[1])

In [ ]:
def generate_from_model(model, tokenizer):
  encoded_input = tokenizer(text, return_tensors='pt')
  output_sequences = model.generate(input_ids=encoded_input['input_ids'].cuda())
  return tokenizer.decode(output_sequences[0], skip_special_tokens=True)

In [ ]:
name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
pipe = pipeline("text-generation",model=name, model_kwargs= {"device_map": "auto"}, max_new_tokens=100, max_length=100)
local_llm = HuggingFacePipeline(pipeline=pipe)
llm_chain = LLMChain(prompt=prompt, 
                     llm=local_llm
                     )
context_only_elab_df = pd.DataFrame()

for text in context_df:
    elab = llm_chain.run(text)
    row = {'context': text, 'elab': elab}
    context_only_elab_df = pd.concat([context_only_elab_df, pd.DataFrame([row])], ignore_index=True)
    
context_only_elab_df.to_csv('DeepSeek-R1-Distill-Qwen-7B+promptEng.csv',index=False)
torch.cuda.empty_cache()

In [ ]:
del pipe
torch.cuda.empty_cache()

In [ ]:
name = "SanketAI/FLAN-T5_instruct-mistral7b"
pipe = pipeline("text2text-generation",model=name, model_kwargs= {"device_map": "auto"}, max_new_tokens=100, max_length=100)
local_llm = HuggingFacePipeline(pipeline=pipe)
llm_chain = LLMChain(prompt=prompt, 
                     llm=local_llm
                     )
context_only_elab_df = pd.DataFrame()

for text in context_df:
    elab = llm_chain.run(text)
    row = {'context': text, 'elab': elab}
    context_only_elab_df = pd.concat([context_only_elab_df, pd.DataFrame([row])], ignore_index=True)
    
context_only_elab_df.to_csv('FLAN-T5_instruct-mistral7b+promptEng.csv',index=False)
torch.cuda.empty_cache()

In [ ]:
del pipe
torch.cuda.empty_cache()